# 查看example_smplx_poses.npy

In [ ]:
# wmj-20260119
# 复现官方example

import numpy as np
import os

def inspect_npy(npy_path, show_values=False, max_items=3, head_n=5):
    """
    Inspect contents of a .npy file (supports dict saved via np.save).
    
    Args:
        npy_path: path to .npy
        show_values: whether to print a small preview of values
        max_items: max number of dict items to preview values for
        head_n: how many elements to show in preview
    """
    assert os.path.exists(npy_path), f"File not found: {npy_path}"
    obj = np.load(npy_path, allow_pickle=True)
    print("========================================")
    print("File:", npy_path)
    print("Loaded type:", type(obj))
    print("========================================")

    # Case 1: saved a dict (common when np.save(path, dict))
    if isinstance(obj, np.ndarray) and obj.dtype == object:
        try:
            data = obj.item()
        except Exception as e:
            print("This looks like an object array but cannot .item() into dict.")
            print("Error:", e)
            print("Array shape:", obj.shape, "dtype:", obj.dtype)
            return

        print("Detected: dict saved in npy")
        print("Keys:", list(data.keys()))
        print("----------------------------------------")
        for k, v in data.items():
            if isinstance(v, np.ndarray):
                print(f"{k:20s} | type=np.ndarray | shape={v.shape} | dtype={v.dtype} | "
                      f"min={np.min(v):.6g} max={np.max(v):.6g}")
            else:
                print(f"{k:20s} | type={type(v)} | value={v}")

        if show_values:
            print("\n========== Value preview ==========")
            for i, (k, v) in enumerate(data.items()):
                if i >= max_items:
                    break
                print(f"\n[{k}]")
                if isinstance(v, np.ndarray):
                    flat = v.reshape(-1)
                    print("head:", flat[:head_n])
                else:
                    print(v)

    # Case 2: saved a plain ndarray
    elif isinstance(obj, np.ndarray):
        arr = obj
        print("Detected: plain numpy array saved in npy")
        print("shape:", arr.shape)
        print("dtype:", arr.dtype)
        print("min:", np.min(arr), "max:", np.max(arr))
        if show_values:
            print("head:", arr.reshape(-1)[:head_n])

    else:
        # Very rare, but handle anyway
        print("Unknown loaded object type.")
        print(obj)

# -------------------------
# Usage:
# -------------------------
npy_path = "data/example_smplx_poses.npy"  # 改成你的文件路径
inspect_npy(npy_path, show_values=True)


File: data/example_smplx_poses.npy
Loaded type: <class 'numpy.ndarray'>
Detected: plain numpy array saved in npy
shape: (8, 165)
dtype: float32
min: -2.6088293 max: 2.9466407
head: [ 1.7001663   0.1067276   0.14347076 -1.2482967  -0.27087525]


# 官方复现

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch
import numpy as np
from mhr.mhr import MHR
from conversion import Conversion
import smplx


def get_parameter_data( input_smplx_poses_file: str,
                        smplx_model: smplx.SMPLX,
                        device: torch.device,):
        """Get SMPL(X) data for examples."""
        # Load the SMPL(X) parameters
        smplx_full_poses = np.load(input_smplx_poses_file)
        num_frames = smplx_full_poses.shape[0]

        smplx_parameters, smpl_parameters = None, None
        # Generate the SMPL(X) parameters by adding random betas and expression coefficients.
        smplx_parameters = {
            "global_orient": np.zeros((num_frames, 3)),
            "body_pose": smplx_full_poses[:, 3:66],
            "left_hand_pose": smplx_full_poses[:, -90:-45],
            "right_hand_pose": smplx_full_poses[:, -45:],
            "betas": np.random.randn(1, smplx_model.num_betas).repeat(
                num_frames, axis=0
            ),
            "expression": np.random.randn(
                1, smplx_model.num_expression_coeffs
            ).repeat(num_frames, axis=0),
        }
        for k, v in smplx_parameters.items():
            smplx_parameters[k] = (
                torch.from_numpy(v).to(torch.float32).to(device)
            )

        smpl_parameters = {
            "global_orient": np.zeros((num_frames, 3)),
            "body_pose": np.concatenate(
                [smplx_full_poses[:, 3:66], np.zeros_like(smplx_full_poses[:, :6])],
                axis=-1,
            ),
            "betas": np.random.randn(num_frames, 10),
        }
        for k, v in smpl_parameters.items():
            smpl_parameters[k] = (
                torch.from_numpy(v).to(torch.float32).to(device)
            )

        return smpl_parameters, smplx_parameters


In [4]:
from example import DEMO
demo=DEMO()
# demo._device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
device = demo._device
print(device)
smplx_model_file="./data/SMPLX_NEUTRAL.npz"

# Initialize models
mhr_model = MHR.from_files(lod=1, device=device)
smplx_model = smplx.SMPLX(model_path=smplx_model_file, gender="neutral", use_pca=False, flat_hand_mean=True).to(device)

smpl_paras, smplx_paras = get_parameter_data(input_smplx_poses_file="./data/example_smplx_poses.npy", smplx_model=smplx_model, device=device)

# Create converter
converter = Conversion(
    mhr_model=mhr_model,
    smpl_model=smplx_model,
    method="pytorch"  # or "pymomentum"
)

print(smpl_paras.keys())
print(smplx_paras.keys())

cpu
dict_keys(['global_orient', 'body_pose', 'betas'])
dict_keys(['global_orient', 'body_pose', 'left_hand_pose', 'right_hand_pose', 'betas', 'expression'])


---

In [5]:

# Convert SMPLX to MHR
mhr_results = converter.convert_smpl2mhr(
    smpl_parameters=smplx_paras,
    single_identity=True,
    return_mhr_meshes=True,
    return_mhr_parameters=True
)



100%|██████████| 1/1 [00:00<00:00, 475.44it/s]
Head pose, identity, and face expression optimization: 100%|██████████| 1/1 [01:39<00:00, 99.53s/it]
100%|██████████| 1/1 [00:00<00:00,  4.96it/s]


**Returns:**
- `ConversionResult`: Object containing requested outputs

### `ConversionResult` Class

Container for conversion results.

**Attributes:**
- `result_meshes` (list[trimesh.Trimesh]): Generated mesh objects
- `result_vertices` (np.ndarray): Vertex positions [B, V, 3]
- `result_parameters` (dict): Model parameter dictionary
- `result_errors` (np.ndarray): Per-frame fitting errors

In [17]:
dir(mhr_results)
mhr_mesh0=mhr_results.result_meshes[0]
mhr_mesh0
# mhr_mesh0.show()
# mhr_mesh0.export("mhr_mesh0.obj") # save

<trimesh.Trimesh(vertices.shape=(18439, 3), faces.shape=(36874, 3))>

In [ ]:
mhr_par0=mhr_results.result_parameters
print(mhr_par0.keys())
for k, v in mhr_par0.items():
    shape = v.shape if hasattr(v, "shape") else None
    print(f"{k:20s} | type={type(v).__name__:15s} | shape={shape}")
    
mhr_results.result_parameters

dict_keys(['lbs_model_params', 'identity_coeffs', 'face_expr_coeffs'])
lbs_model_params     | type=Tensor          | shape=torch.Size([8, 204])
identity_coeffs      | type=Tensor          | shape=torch.Size([8, 45])
face_expr_coeffs     | type=Tensor          | shape=torch.Size([8, 72])


{'lbs_model_params': tensor([[ 2.3482e-02, -1.3065e+01, -3.7516e-01,  ..., -9.9845e-02,
          -1.4634e-01,  8.5001e-02],
         [-2.7500e-03, -1.3128e+01, -3.1339e-01,  ..., -9.9845e-02,
          -1.4634e-01,  8.5001e-02],
         [ 3.8325e-02, -1.3104e+01, -3.6292e-01,  ..., -9.9845e-02,
          -1.4634e-01,  8.5001e-02],
         ...,
         [-9.8679e-02, -1.3040e+01, -5.2069e-01,  ..., -9.9845e-02,
          -1.4634e-01,  8.5001e-02],
         [-1.5999e-02, -1.3142e+01, -3.2818e-01,  ..., -9.9845e-02,
          -1.4634e-01,  8.5001e-02],
         [ 9.2094e-03, -1.3104e+01, -3.8024e-01,  ..., -9.9845e-02,
          -1.4634e-01,  8.5001e-02]], grad_fn=<CopySlices>),
 'identity_coeffs': tensor([[-0.3374,  0.9505, -2.2492,  1.1178,  0.8396, -1.5504,  0.6886, -3.0673,
           0.5778,  2.7482, -5.6396,  1.5769, -4.7640,  3.3278,  2.0701,  1.4990,
           2.4834,  1.9388, -3.6560, -2.2268, -0.5562, -0.6463,  1.2167,  1.0492,
          -0.5596, -1.2104, -0.6084,  0.7607, -

---

In [8]:
# Convert MHR back to SMPLX
smplx_results = converter.convert_mhr2smpl(
    mhr_parameters=mhr_results.result_parameters,
    return_smpl_meshes=True
)

100%|██████████| 1/1 [00:00<00:00, 11.09it/s]


In [16]:
smplx_mesh0=smplx_results.result_meshes[0]
smplx_mesh0
# smplx_mesh0.show()

<trimesh.Trimesh(vertices.shape=(10475, 3), faces.shape=(20908, 3))>

In [10]:
smplx_par0=smplx_results.result_parameters
for k, v in smplx_par0.items():
    shape = v.shape if hasattr(v, "shape") else None
    print(f"{k:20s} | type={type(v).__name__:15s} | shape={shape}")

global_orient        | type=Tensor          | shape=torch.Size([8, 3])
transl               | type=Tensor          | shape=torch.Size([8, 3])
body_pose            | type=Tensor          | shape=torch.Size([8, 63])
betas                | type=Tensor          | shape=torch.Size([8, 10])
left_hand_pose       | type=Tensor          | shape=torch.Size([8, 45])
right_hand_pose      | type=Tensor          | shape=torch.Size([8, 45])
expression           | type=Tensor          | shape=torch.Size([8, 10])
